# Optimisation des modèles — `/predict`

**Objectif :** vérifier si un modèle non linéaire bien réglé peut faire mieux que la régression linéaire.

Démarche : tuning léger de cinq familles de modèles, tuning approfondi des deux meilleures, puis comparaison avec la régression linéaire. Validation croisée à 5 folds sur le train ; le jeu de test reste réservé à l'évaluation finale.

## Imports

In [1]:
import time

import pandas as pd
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBRegressor

from agritech.config import SEED
from agritech.evaluation import CV_SCORING, cross_validate_folds, format_cv_metrics, summarize_cv_folds
from agritech.notification import notify
from agritech.preprocessing import PREPROCESSING_DESCRIPTION, build_pipeline, count_encoded_columns
from agritech.tracking import log_run, run_tags, setup_mlflow
from agritech.training_data import load_predict_dataset, predict_cv, predict_feature_types, predict_protocol_params, split_predict

# Données et protocole

On utilise uniquement les 4 variables retenues dans le notebook 08b : pluie, température, fertilisation et irrigation. `Crop` reste une information métier de l'application, mais le modèle actuel ne l'utilise pas.

In [2]:
VARIABLES = ["Rainfall_mm", "Temperature_Celsius", "Fertilizer_Used", "Irrigation_Used"]

df = load_predict_dataset()
X_train, X_test, y_train, y_test = split_predict(df)
cv = predict_cv()
categorielles, numeriques = predict_feature_types(VARIABLES)

print(f"X_train : {X_train.shape} | X_test : {X_test.shape}, réservé à l'évaluation finale")
print(f"variables : {VARIABLES} | colonnes après encodage : {count_encoded_columns(categorielles, numeriques, X_train)}")

X_train : (799815, 9) | X_test : (199954, 9), réservé à l'évaluation finale
variables : ['Rainfall_mm', 'Temperature_Celsius', 'Fertilizer_Used', 'Irrigation_Used'] | colonnes après encodage : 6


In [3]:
# un tag d'étape par phase de tuning, pour les distinguer dans MLflow
experience = setup_mlflow("predict")
TAGS = {
    "light": run_tags(service="predict", stage="light_tuning", notebook="09_predict_model_tuning.ipynb"),
    "deep": run_tags(service="predict", stage="deep_tuning", notebook="09_predict_model_tuning.ipynb"),
}
PARAMS_PROTOCOLE = predict_protocol_params(X_train, X_test)

expérience MLflow : oc_p12_agritech_predict


## Fonctions communes

In [4]:
def valider(modele, libelle):
    """Validation croisée d'un modèle sur les 4 variables : scores par fold et résumé."""
    debut = time.perf_counter()
    folds = cross_validate_folds(build_pipeline(modele, categorielles, numeriques), X_train[VARIABLES], y_train, cv)
    metriques = summarize_cv_folds(folds, with_fit_time=True)
    metriques["cv_total_time"] = time.perf_counter() - debut

    print(f"{libelle:32s} {format_cv_metrics(metriques)} | entraînement {metriques['cv_fit_time_mean']:.1f} s")
    return folds, metriques


def journaliser(nom_run, phase, modele, parametres, metriques):
    """Run MLflow d'une configuration : protocole, modèle, phase de tuning, hyperparamètres et métriques."""
    log_run(
        nom_run,
        TAGS[phase],
        params=PARAMS_PROTOCOLE | {
            "model": type(modele).__name__,
            "tuning_phase": phase,
            "feature_set": "useful_4",
            "n_features": len(VARIABLES),
            "preprocessing": PREPROCESSING_DESCRIPTION,
        } | parametres,
        metrics=metriques,
    )


def rechercher(nom, prefixe, modele, espace, n_essais, phase):
    """Recherche aléatoire sur les mêmes 5 folds, puis un run MLflow par configuration testée."""
    recherche = RandomizedSearchCV(
        build_pipeline(modele, categorielles, numeriques),
        {f"model__{cle}": valeurs for cle, valeurs in espace.items()},  # « model__ » désigne le modèle dans le pipeline
        n_iter=n_essais,
        scoring=CV_SCORING,
        refit=False,  # les meilleures configurations sont réévaluées plus bas
        cv=cv,
        random_state=SEED,
        n_jobs=1,  # le modèle utilise déjà tous les cœurs
        error_score="raise",
    )
    debut = time.perf_counter()
    recherche.fit(X_train[VARIABLES], y_train)
    duree = time.perf_counter() - debut

    # scikit-learn renvoie la RMSE et la MAE en négatif, car ses scores suivent la règle « plus grand = meilleur »
    brut = pd.DataFrame(recherche.cv_results_)
    essais = pd.DataFrame({cle: brut[f"param_model__{cle}"] for cle in espace})
    essais["cv_rmse_mean"] = -brut["mean_test_rmse"]
    essais["cv_rmse_std"] = brut["std_test_rmse"]
    essais["cv_mae_mean"] = -brut["mean_test_mae"]
    essais["cv_mae_std"] = brut["std_test_mae"]
    essais["cv_r2_mean"] = brut["mean_test_r2"]
    essais["cv_r2_std"] = brut["std_test_r2"]
    essais["cv_fit_time_mean"] = brut["mean_fit_time"]
    essais["cv_fit_time_std"] = brut["std_fit_time"]
    # hyperparamètres de chaque essai, sans le préfixe « model__ », pour MLflow et pour recréer le modèle
    essais["parametres"] = [{cle.removeprefix("model__"): valeur for cle, valeur in dico.items()} for dico in brut["params"]]

    metriques = [colonne for colonne in essais.columns if colonne.startswith("cv_")]
    for numero in essais.index:
        journaliser(f"{prefixe}_{phase}_{numero + 1:02d}", phase, modele, essais.at[numero, "parametres"],
                    {colonne: float(essais.at[numero, colonne]) for colonne in metriques})

    print(f"{nom:22s} {n_essais} configurations en {duree / 60:4.1f} min | meilleure RMSE {essais['cv_rmse_mean'].min():.6f}")
    return essais.sort_values("cv_rmse_mean"), duree


def position_dans_la_plage(essais, espace):
    """Pour la meilleure configuration : valeur de chaque hyperparamètre et position dans les valeurs testées."""
    meilleurs = essais.iloc[0]["parametres"]
    lignes = []
    for cle, valeurs in espace.items():
        valeur = meilleurs[cle]
        if valeur == valeurs[0]:
            position = "début de plage"
        elif valeur == valeurs[-1]:
            position = "fin de plage"
        else:
            position = "intérieur"
        # texte plutôt que nombre : pandas afficherait sinon les entiers avec une décimale
        lignes.append({"hyperparamètre": cle, "valeur retenue": str(valeur), "valeurs testées": valeurs, "position": position})
    return pd.DataFrame(lignes).set_index("hyperparamètre")

# Régression linéaire de référence

Recalculée pour avoir les scores par fold, qui serviront à la comparaison finale. Elle est déjà journalisée dans MLflow par le notebook 08b.

In [5]:
folds_reference, metriques_reference = valider(LinearRegression(), "LinearRegression")

LinearRegression                 RMSE 0.5003 ± 0.0009 t/ha | MAE 0.3993 ± 0.0007 t/ha | R² 0.9129 ± 0.0003 | entraînement 0.1 s


# Tuning léger

Cinq familles de modèles, 10 configurations chacune, tirées au hasard parmi quelques valeurs d'hyperparamètres. Toutes les configurations sont évaluées sur les mêmes 5 folds.

In [6]:
MODELES_LEGERS = {
    "RandomForest": ("random_forest", RandomForestRegressor(n_jobs=-1, random_state=SEED), {
        "n_estimators": [100, 300],
        "max_depth": [10, 20, None],
        "min_samples_leaf": [20, 100, 500],
        "max_features": [0.5, 1.0],
    }),
    "HistGradientBoosting": ("hgb", HistGradientBoostingRegressor(early_stopping=False, random_state=SEED), {
        "learning_rate": [0.03, 0.1, 0.3],
        "max_iter": [100, 300, 1000],
        "max_leaf_nodes": [7, 31, 63],
        "min_samples_leaf": [20, 200, 1000],
        "l2_regularization": [0.0, 1.0],
    }),
    "XGBoost": ("xgboost", XGBRegressor(random_state=SEED), {
        "learning_rate": [0.03, 0.1, 0.3],
        "n_estimators": [100, 300, 1000],
        "max_depth": [3, 6, 9],
        "min_child_weight": [1, 100, 1000],
        "subsample": [0.8, 1.0],
    }),
    "LightGBM": ("lightgbm", LGBMRegressor(random_state=SEED, verbose=-1), {
        "learning_rate": [0.03, 0.1, 0.3],
        "n_estimators": [100, 300, 1000],
        "num_leaves": [7, 31, 63],
        "min_child_samples": [20, 200, 1000],
        "reg_lambda": [0.0, 1.0],
    }),
    "CatBoost": ("catboost", CatBoostRegressor(random_seed=SEED, verbose=0, allow_writing_files=False), {
        "learning_rate": [0.03, 0.1, 0.3],
        "iterations": [300, 1000],
        "depth": [4, 6, 8],
        "l2_leaf_reg": [1, 3, 10],
    }),
}
N_ESSAIS_LEGERS = 10

In [7]:
resultats_legers = {}
durees_legeres = {}
for nom, (prefixe, modele, espace) in MODELES_LEGERS.items():
    resultats_legers[nom], durees_legeres[nom] = rechercher(nom, prefixe, modele, espace, N_ESSAIS_LEGERS, "light")

RandomForest           10 configurations en  8.2 min | meilleure RMSE 0.501167


HistGradientBoosting   10 configurations en  6.2 min | meilleure RMSE 0.500737


XGBoost                10 configurations en  2.3 min | meilleure RMSE 0.501176


LightGBM               10 configurations en  3.0 min | meilleure RMSE 0.500895


CatBoost               10 configurations en  4.0 min | meilleure RMSE 0.500503


In [8]:
lignes = []
for nom, essais in resultats_legers.items():
    meilleur = essais.iloc[0]
    lignes.append({
        "modèle": nom,
        "meilleure RMSE": meilleur["cv_rmse_mean"],
        "R²": meilleur["cv_r2_mean"],
        "entraînement (s)": meilleur["cv_fit_time_mean"],
        "recherche (min)": durees_legeres[nom] / 60,
        "écart avec la régression linéaire": meilleur["cv_rmse_mean"] - metriques_reference["cv_rmse_mean"],
    })

synthese_legere = pd.DataFrame(lignes).set_index("modèle").sort_values("meilleure RMSE")
print(f"régression linéaire : RMSE {metriques_reference['cv_rmse_mean']:.6f} | R² {metriques_reference['cv_r2_mean']:.4f}")
synthese_legere.round({"meilleure RMSE": 6, "R²": 4, "entraînement (s)": 1, "recherche (min)": 1, "écart avec la régression linéaire": 6})

régression linéaire : RMSE 0.500332 | R² 0.9129


,meilleure RMSE,R²,entraînement (s),recherche (min),écart avec la régression linéaire
modèle,,,,,
CatBoost,0.500503,0.9128,5.0,4.0,0.000172
HistGradientBoosting,0.500737,0.9127,6.4,6.2,0.000406
LightGBM,0.500895,0.9127,5.1,3.0,0.000564
RandomForest,0.501167,0.9126,14.7,8.2,0.000836
XGBoost,0.501176,0.9126,3.4,2.3,0.000845


**Observations :**

- Aucun modèle ne bat la régression linéaire.
- CatBoost et HistGradientBoosting sont les deux meilleurs modèles non linéaires : ils passent au tuning approfondi.

# Sélection des finalistes

Pour chaque finaliste, on regarde si les meilleures valeurs sont aux limites des valeurs testées (début ou fin de plage) : dans ce cas, le tuning suivant élargit la recherche.

In [9]:
finalistes = list(synthese_legere.index[:2])
print("finalistes :", finalistes)

finalistes : ['CatBoost', 'HistGradientBoosting']


In [10]:
for nom in finalistes:
    print(f"\n{nom} — meilleure configuration du tuning léger")
    display(position_dans_la_plage(resultats_legers[nom], MODELES_LEGERS[nom][2]))


CatBoost — meilleure configuration du tuning léger


,valeur retenue,valeurs testées,position
hyperparamètre,,,
learning_rate,0.03,"[0.03, 0.1, 0.3]",début de plage
iterations,1000,"[300, 1000]",fin de plage
depth,4,"[4, 6, 8]",début de plage
l2_leaf_reg,3,"[1, 3, 10]",intérieur



HistGradientBoosting — meilleure configuration du tuning léger


,valeur retenue,valeurs testées,position
hyperparamètre,,,
learning_rate,0.03,"[0.03, 0.1, 0.3]",début de plage
max_iter,1000,"[100, 300, 1000]",fin de plage
max_leaf_nodes,7,"[7, 31, 63]",début de plage
min_samples_leaf,1000,"[20, 200, 1000]",fin de plage
l2_regularization,1.0,"[0.0, 1.0]",fin de plage


**Observations :**

- Plusieurs meilleures valeurs sont aux limites des valeurs testées, notamment la vitesse d'apprentissage la plus basse et le nombre d'arbres le plus élevé.
- Le tuning approfondi élargit donc les plages dans ces directions.

# Tuning approfondi

35 configurations pour chacun des deux finalistes, avec des plages élargies à partir des résultats du tuning léger.

In [11]:
ESPACES_APPROFONDIS = {
    "CatBoost": ("catboost", CatBoostRegressor(random_seed=SEED, verbose=0, allow_writing_files=False), {
        "learning_rate": [0.005, 0.01, 0.02, 0.03],
        "iterations": [1000, 2000, 3000, 5000],
        "depth": [2, 3, 4, 5, 6],
        "l2_leaf_reg": [0.3, 1, 3, 10, 30],
    }),
    "HistGradientBoosting": ("hgb", HistGradientBoostingRegressor(early_stopping=False, random_state=SEED), {
        "learning_rate": [0.005, 0.01, 0.02, 0.03],
        "max_iter": [500, 1000, 2000, 3000],
        "max_leaf_nodes": [5, 7, 15, 31],
        "min_samples_leaf": [200, 500, 1000, 2000, 5000],
        "l2_regularization": [0.5, 1.0, 5.0, 10.0],
    }),
}
N_ESSAIS_APPROFONDIS = 35

In [12]:
resultats_approfondis = {}
durees_approfondies = {}
for nom in finalistes:
    prefixe, modele, espace = ESPACES_APPROFONDIS[nom]
    resultats_approfondis[nom], durees_approfondies[nom] = rechercher(nom, prefixe, modele, espace, N_ESSAIS_APPROFONDIS, "deep")

CatBoost               35 configurations en 41.8 min | meilleure RMSE 0.500411


HistGradientBoosting   35 configurations en 50.0 min | meilleure RMSE 0.500640


Cinq meilleures configurations de chaque finaliste, puis position de la meilleure dans les valeurs testées.

In [13]:
COLONNES_APPROFONDIES = ["cv_rmse_mean", "cv_rmse_std", "cv_r2_mean", "cv_fit_time_mean"]

for nom in finalistes:
    espace = ESPACES_APPROFONDIS[nom][2]
    print(f"\n{nom} — {N_ESSAIS_APPROFONDIS} configurations en {durees_approfondies[nom] / 60:.0f} min")
    display(resultats_approfondis[nom][list(espace) + COLONNES_APPROFONDIES].head(5).round({"cv_rmse_mean": 6, "cv_rmse_std": 6, "cv_r2_mean": 4, "cv_fit_time_mean": 1}))
    display(position_dans_la_plage(resultats_approfondis[nom], espace))


CatBoost — 35 configurations en 42 min


,learning_rate,iterations,depth,l2_leaf_reg,cv_rmse_mean,cv_rmse_std,cv_r2_mean,cv_fit_time_mean
1,0.005,3000,5,0.3,0.500411,0.000886,0.9129,17.1
34,0.005,5000,4,10.0,0.500424,0.000892,0.9129,24.5
17,0.005,3000,6,30.0,0.500425,0.000884,0.9129,19.6
14,0.005,3000,3,10.0,0.500426,0.000889,0.9129,13.0
13,0.005,5000,2,10.0,0.500431,0.000885,0.9129,18.5


,valeur retenue,valeurs testées,position
hyperparamètre,,,
learning_rate,0.005,"[0.005, 0.01, 0.02, 0.03]",début de plage
iterations,3000,"[1000, 2000, 3000, 5000]",intérieur
depth,5,"[2, 3, 4, 5, 6]",intérieur
l2_leaf_reg,0.3,"[0.3, 1, 3, 10, 30]",début de plage



HistGradientBoosting — 35 configurations en 50 min


,learning_rate,max_iter,max_leaf_nodes,min_samples_leaf,l2_regularization,cv_rmse_mean,cv_rmse_std,cv_r2_mean,cv_fit_time_mean
26,0.005,3000,15,5000,1.0,0.500640,0.000890,0.9128,32.6
33,0.010,2000,7,2000,0.5,0.500663,0.000865,0.9128,13.3
3,0.010,2000,15,200,5.0,0.500667,0.000888,0.9128,21.4
23,0.020,1000,15,1000,5.0,0.500680,0.000882,0.9128,10.9
11,0.030,500,15,1000,1.0,0.500683,0.000879,0.9128,5.4


,valeur retenue,valeurs testées,position
hyperparamètre,,,
learning_rate,0.005,"[0.005, 0.01, 0.02, 0.03]",début de plage
max_iter,3000,"[500, 1000, 2000, 3000]",fin de plage
max_leaf_nodes,15,"[5, 7, 15, 31]",intérieur
min_samples_leaf,5000,"[200, 500, 1000, 2000, 5000]",fin de plage
l2_regularization,1.0,"[0.5, 1.0, 5.0, 10.0]",intérieur


**Observations :**

- Les deux finalistes s'améliorent légèrement, mais restent derrière la régression linéaire.
- Pour chaque modèle, les meilleures configurations donnent des RMSE très proches.
- Certaines meilleures valeurs restent aux limites des plages, mais poursuivre le tuning coûterait davantage de temps pour un gain minime : on s'arrête là.

# Meilleures configurations réévaluées

La meilleure configuration de chaque finaliste est réévaluée avec les mêmes 5 folds, pour obtenir ses scores par fold.

In [14]:
candidats = {}
for nom in finalistes:
    prefixe, modele, espace = ESPACES_APPROFONDIS[nom]
    meilleurs = resultats_approfondis[nom].iloc[0]["parametres"]
    print(f"{nom} : {meilleurs}")
    candidats[nom] = valider(clone(modele).set_params(**meilleurs), nom)

CatBoost : {'learning_rate': 0.005, 'l2_leaf_reg': 0.3, 'iterations': 3000, 'depth': 5}


CatBoost                         RMSE 0.5004 ± 0.0009 t/ha | MAE 0.3994 ± 0.0007 t/ha | R² 0.9129 ± 0.0003 | entraînement 17.2 s
HistGradientBoosting : {'min_samples_leaf': 5000, 'max_leaf_nodes': 15, 'max_iter': 3000, 'learning_rate': 0.005, 'l2_regularization': 1.0}


HistGradientBoosting             RMSE 0.5006 ± 0.0009 t/ha | MAE 0.3995 ± 0.0007 t/ha | R² 0.9128 ± 0.0003 | entraînement 31.7 s


**Observations :**

- Les scores réévalués sont identiques à ceux de la recherche : les résultats sont reproductibles.
- CatBoost reste le meilleur finaliste, devant HistGradientBoosting.

# Comparaison avec la régression linéaire

RMSE fold par fold, puis écart avec la régression linéaire. Un écart négatif signifie que le modèle fait mieux.

In [15]:
rmse_par_fold = pd.DataFrame({"régression linéaire": folds_reference["rmse"]})
for nom, (folds, _) in candidats.items():
    rmse_par_fold[nom] = folds["rmse"]
rmse_par_fold.round(6)

,régression linéaire,CatBoost,HistGradientBoosting
fold,,,
1,0.499270,0.499329,0.499510
2,0.499878,0.499966,0.500208
3,0.499747,0.499867,0.500138
4,0.501126,0.501199,0.501427
5,0.501638,0.501693,0.501917


In [16]:
lignes = [{
    "modèle": "LinearRegression",
    "RMSE": metriques_reference["cv_rmse_mean"],
    "MAE": metriques_reference["cv_mae_mean"],
    "R²": metriques_reference["cv_r2_mean"],
    "écart moyen": 0.0,
    "folds gagnés": "—",  # la référence ne se compare pas à elle-même
    "entraînement (s)": metriques_reference["cv_fit_time_mean"],
}]
for nom, (folds, metriques) in candidats.items():
    ecart = folds["rmse"] - folds_reference["rmse"]
    lignes.append({
        "modèle": nom,
        "RMSE": metriques["cv_rmse_mean"],
        "MAE": metriques["cv_mae_mean"],
        "R²": metriques["cv_r2_mean"],
        "écart moyen": ecart.mean(),
        "folds gagnés": int((ecart < 0).sum()),
        "entraînement (s)": metriques["cv_fit_time_mean"],
    })

print(f"écart-type de la RMSE entre folds, régression linéaire : {metriques_reference['cv_rmse_std']:.6f} t/ha")
pd.DataFrame(lignes).set_index("modèle").round({"RMSE": 6, "MAE": 6, "R²": 5, "écart moyen": 6, "entraînement (s)": 2})

écart-type de la RMSE entre folds, régression linéaire : 0.000896 t/ha


,RMSE,MAE,R²,écart moyen,folds gagnés,entraînement (s)
modèle,,,,,,
LinearRegression,0.500332,0.399292,0.91289,0.000000,—,0.11
CatBoost,0.500411,0.399369,0.91286,0.000079,0,17.20
HistGradientBoosting,0.500640,0.399543,0.91278,0.000308,0,31.71


**Observations :**

- La régression linéaire fait mieux que les deux finalistes sur chacun des 5 folds.
- Les écarts sont très faibles, plus petits que la variation d'un fold à l'autre, mais toujours en sa faveur.
- Elle s'entraîne aussi beaucoup plus vite : 0,1 s, contre 17 s pour CatBoost et 32 s pour HistGradientBoosting.

# Conclusion

**Observations :**

- la régression linéaire obtient la meilleure RMSE en validation croisée : 0,500332 t/ha ;
- CatBoost optimisé s'en approche à 0,00008 t/ha, mais ne fait mieux sur aucun fold ;
- les modèles non linéaires ajoutent de la complexité sans améliorer les résultats, même après deux phases de tuning ;
- modèle retenu pour `/predict` : la régression linéaire, rapide à entraîner et dont les coefficients se lisent directement.

Le jeu de test reste réservé à l'évaluation finale.

In [17]:
resume = [f"{nom} {metriques['cv_rmse_mean']:.6f}" for nom, (_, metriques) in candidats.items()]
notify("Agritech /predict", f"Tuning terminé — RMSE CV : {', '.join(resume)} | régression linéaire {metriques_reference['cv_rmse_mean']:.6f}")